# Notebook 00: Platform Setup & Data Foundation

**Persona:** Database Administrator (DBA) / Platform Owner

**Purpose:** Create the complete infrastructure for the Feature Store implementation guide:
- Custom roles with hierarchical RBAC
- Separate databases for development and production
- Clickstream source data tables
- Synthetic test data at configurable scale
- Dev branch with isolated copy of data

**Prerequisites:** ACCOUNTADMIN or SYSADMIN access

## 1. Configuration

All environment parameters are centralised in `feature_definitions/config.py`.
Override any value with environment variables prefixed `FS_`.

In [ ]:
import os
import sys

# -- Workspace Notebook vs local detection --
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    IN_WORKSPACE = True
except Exception:
    IN_WORKSPACE = False

# Ensure feature_definitions is importable
if not IN_WORKSPACE:
    sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
    sys.path.insert(0, ".")

from feature_definitions.config import (
    get_config, get_session, ROLES, DATA_SCALE, ENVIRONMENTS,
)

print(f"Running in {'Workspace Notebook' if IN_WORKSPACE else 'local'} mode")
print(f"Data scale: {DATA_SCALE}")
print(f"Dev DB:  {ENVIRONMENTS['DEV']['database']}")
print(f"Prod DB: {ENVIRONMENTS['PROD']['database']}")

## 2. Connect & Assume ACCOUNTADMIN

Role and database creation requires ACCOUNTADMIN (or a role with CREATE ROLE + CREATE DATABASE privileges). If running in Workspace, this cell uses the active session.

In [ ]:
if IN_WORKSPACE:
    session.sql("USE ROLE ACCOUNTADMIN").collect()
else:
    session = get_session(role="ACCOUNTADMIN")

print(f"Connected as {session.get_current_user()} / {session.get_current_role()}")

prod_cfg = get_config("PROD")
dev_cfg  = get_config("DEV")
admin_role    = ROLES["admin"]
dev_role      = ROLES["dev"]
consumer_role = ROLES["consumer"]
warehouse     = prod_cfg["warehouse"]
wh_size       = prod_cfg["warehouse_size"]
prod_db       = prod_cfg["database"]
dev_db        = dev_cfg["database"]

## 3. Create Roles

The role hierarchy follows the Feature Store best practice pattern:

```
ACCOUNTADMIN
  └── SYSADMIN
        └── FS_ADMIN_ROLE   (production deployment)
              └── FS_DEV_ROLE   (feature engineering)
                    └── FS_CONSUMER_ROLE (read-only)
```

In [ ]:
for role_name, comment in [
    (consumer_role, "Read-only access to feature store"),
    (dev_role,      "Feature store development - create DT-based features"),
    (admin_role,    "Feature store admin - production deployment and promotion"),
]:
    session.sql(f"CREATE ROLE IF NOT EXISTS {role_name} COMMENT = '{comment}'").collect()
    print(f"  Created: {role_name}")

# Establish hierarchy
session.sql(f"GRANT ROLE {consumer_role} TO ROLE {dev_role}").collect()
session.sql(f"GRANT ROLE {dev_role}      TO ROLE {admin_role}").collect()
session.sql(f"GRANT ROLE {admin_role}    TO ROLE SYSADMIN").collect()

# Grant all roles to current user
current_user = session.get_current_user().strip('"')
for r in [consumer_role, dev_role, admin_role]:
    session.sql(f"GRANT ROLE {r} TO USER {current_user}").collect()

print("\nRole hierarchy established")

## 4. Create Warehouse

A single X-SMALL warehouse is sufficient for the demo. Override `FS_DEV_WAREHOUSE_SIZE` for larger tests.

In [ ]:
session.sql(f"""
    CREATE WAREHOUSE IF NOT EXISTS {warehouse}
        WAREHOUSE_SIZE = '{wh_size}'
        AUTO_SUSPEND = 60
        AUTO_RESUME = TRUE
        INITIALLY_SUSPENDED = TRUE
        COMMENT = 'Feature Store Implementation Guide'
""").collect()

session.sql(f"GRANT USAGE   ON WAREHOUSE {warehouse} TO ROLE {consumer_role}").collect()
session.sql(f"GRANT USAGE   ON WAREHOUSE {warehouse} TO ROLE {dev_role}").collect()
session.sql(f"GRANT OPERATE ON WAREHOUSE {warehouse} TO ROLE {admin_role}").collect()
print(f"Warehouse {warehouse} ({wh_size}) ready")

## 5. Create Databases

| Database | Purpose | Owner |
|---|---|---|
| `FEATURE_STORE_DEMO` | Production features, models, inference | FS_ADMIN_ROLE |
| `FEATURE_STORE_DEMO_DEV` | Development sandbox | FS_DEV_ROLE |

In [ ]:
for db_name in [prod_db, dev_db]:
    session.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}").collect()

# Ownership
session.sql(f"GRANT OWNERSHIP ON DATABASE {prod_db} TO ROLE {admin_role} COPY CURRENT GRANTS").collect()
session.sql(f"GRANT CREATE SCHEMA ON DATABASE {prod_db} TO ROLE {admin_role}").collect()
session.sql(f"GRANT OWNERSHIP ON DATABASE {dev_db} TO ROLE {dev_role} COPY CURRENT GRANTS").collect()
session.sql(f"GRANT CREATE SCHEMA ON DATABASE {dev_db} TO ROLE {dev_role}").collect()

# Cross-env read access
session.sql(f"GRANT USAGE ON DATABASE {prod_db} TO ROLE {dev_role}").collect()
session.sql(f"GRANT USAGE ON DATABASE {prod_db} TO ROLE {consumer_role}").collect()
session.sql(f"GRANT USAGE ON DATABASE {dev_db}  TO ROLE {consumer_role}").collect()

# Task execution privileges (needed for DT management)
for role in [admin_role, dev_role]:
    session.sql(f"GRANT EXECUTE TASK ON ACCOUNT TO ROLE {role}").collect()
    session.sql(f"GRANT EXECUTE MANAGED TASK ON ACCOUNT TO ROLE {role}").collect()

print(f"Databases created: {prod_db}, {dev_db}")

## 6. Create Schemas

Each database gets a consistent set of schemas separating source data, features, training artifacts, and inference outputs.

In [ ]:
session.sql(f"USE ROLE {admin_role}").collect()
session.sql(f"USE WAREHOUSE {warehouse}").collect()

schemas = {
    "source_schema":      "Clickstream source data tables",
    "admin_schema":       "Incremental generator configuration",
    "fs_schema":          "Feature store entities and feature views",
    "ml_datasets_schema": "Public ML datasets",
    "training_schema":    "Generated training datasets",
    "inference_schema":   "Batch inference inputs and outputs",
    "spines_schema":      "Entity-timestamp spine tables",
}

# Production schemas
for key, comment in schemas.items():
    schema_name = prod_cfg[key]
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {prod_db}.{schema_name} COMMENT = '{comment}'").collect()
    print(f"  {prod_db}.{schema_name}")

# Future grants for downstream roles
for role in [dev_role, consumer_role]:
    session.sql(f"GRANT USAGE ON ALL SCHEMAS IN DATABASE {prod_db} TO ROLE {role}").collect()
    session.sql(f"GRANT USAGE ON FUTURE SCHEMAS IN DATABASE {prod_db} TO ROLE {role}").collect()
    session.sql(f"GRANT SELECT ON FUTURE TABLES IN DATABASE {prod_db} TO ROLE {role}").collect()
    session.sql(f"GRANT SELECT ON FUTURE VIEWS IN DATABASE {prod_db} TO ROLE {role}").collect()
    session.sql(f"GRANT SELECT ON FUTURE DYNAMIC TABLES IN DATABASE {prod_db} TO ROLE {role}").collect()

## 7. Create Source Data Tables

The Clickstream data model includes 11 tables spanning customers, products, sessions, events, and orders. See `CLICKSTREAM_DATA_MODEL.md` for the full ERD.

In [ ]:
src = prod_cfg["source_schema"]
session.sql(f"USE SCHEMA {prod_db}.{src}").collect()

table_ddl = {
    "CATEGORIES": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.CATEGORIES (
        CATEGORY_ID VARCHAR(20) PRIMARY KEY,
        CATEGORY_NAME VARCHAR(100) NOT NULL,
        PARENT_CATEGORY_ID VARCHAR(20),
        LEVEL INT DEFAULT 1,
        PATH VARCHAR(500),
        IS_ACTIVE BOOLEAN DEFAULT TRUE,
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "SUPPLIERS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.SUPPLIERS (
        SUPPLIER_ID VARCHAR(20) PRIMARY KEY,
        SUPPLIER_NAME VARCHAR(200) NOT NULL,
        CONTACT_EMAIL VARCHAR(200),
        CONTACT_PHONE VARCHAR(50),
        ADDRESS VARCHAR(500),
        CITY VARCHAR(100),
        STATE VARCHAR(50),
        COUNTRY VARCHAR(100) DEFAULT 'USA',
        IS_ACTIVE BOOLEAN DEFAULT TRUE,
        RATING DECIMAL(3,2),
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "PRODUCTS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.PRODUCTS (
        PRODUCT_ID VARCHAR(20) PRIMARY KEY,
        PRODUCT_NAME VARCHAR(200) NOT NULL,
        CATEGORY_ID VARCHAR(20),
        BRAND VARCHAR(100),
        DESCRIPTION VARCHAR(2000),
        BASE_PRICE DECIMAL(10,2),
        CURRENT_PRICE DECIMAL(10,2),
        COST DECIMAL(10,2),
        SKU VARCHAR(50),
        WEIGHT_KG DECIMAL(8,3),
        IS_ACTIVE BOOLEAN DEFAULT TRUE,
        LAUNCH_DATE DATE,
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "PRODUCT_SUPPLIER": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.PRODUCT_SUPPLIER (
        PRODUCT_ID VARCHAR(20),
        SUPPLIER_ID VARCHAR(20),
        IS_PRIMARY BOOLEAN DEFAULT FALSE,
        LEAD_TIME_DAYS INT,
        UNIT_COST DECIMAL(10,2),
        MIN_ORDER_QTY INT DEFAULT 1,
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        PRIMARY KEY (PRODUCT_ID, SUPPLIER_ID)
    )""",
    "HOUSEHOLDS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.HOUSEHOLDS (
        HOUSEHOLD_ID VARCHAR(20) PRIMARY KEY,
        CITY VARCHAR(100), STATE VARCHAR(50),
        COUNTRY VARCHAR(100) DEFAULT 'USA',
        HOUSEHOLD_SIZE INT, INCOME_BRACKET VARCHAR(50),
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "VISITORS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.VISITORS (
        VISITOR_ID VARCHAR(40) PRIMARY KEY,
        FIRST_SEEN_TS TIMESTAMP_NTZ NOT NULL,
        LAST_SEEN_TS TIMESTAMP_NTZ,
        FIRST_DEVICE_TYPE VARCHAR(50),
        FIRST_UTM_SOURCE VARCHAR(100),
        SESSION_CNT INT DEFAULT 0,
        PAGE_VIEW_CNT INT DEFAULT 0,
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "USERS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.USERS (
        USER_ID VARCHAR(40) PRIMARY KEY,
        VISITOR_ID VARCHAR(40),
        HOUSEHOLD_ID VARCHAR(20),
        EMAIL VARCHAR(200),
        FIRST_NAME VARCHAR(100), LAST_NAME VARCHAR(100),
        REGISTRATION_TS TIMESTAMP_NTZ NOT NULL,
        LAST_LOGIN_TS TIMESTAMP_NTZ,
        IS_ACTIVE BOOLEAN DEFAULT TRUE,
        EMAIL_VERIFIED BOOLEAN DEFAULT FALSE,
        LOYALTY_TIER VARCHAR(50) DEFAULT 'Bronze',
        LOYALTY_POINTS INT DEFAULT 0,
        PREFERRED_LANGUAGE VARCHAR(10) DEFAULT 'en',
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "SESSIONS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.SESSIONS (
        SESSION_ID VARCHAR(40) PRIMARY KEY,
        VISITOR_ID VARCHAR(40) NOT NULL,
        USER_ID VARCHAR(40),
        SESSION_START_TS TIMESTAMP_NTZ NOT NULL,
        SESSION_END_TS TIMESTAMP_NTZ,
        DURATION_SEC INT, EVENT_CNT INT DEFAULT 0,
        PAGE_VIEW_CNT INT DEFAULT 0,
        PRODUCT_VIEW_DCNT INT DEFAULT 0,
        CART_ADD_CNT INT DEFAULT 0,
        CART_VALUE_SUM DECIMAL(12,2) DEFAULT 0,
        IS_CONVERTED BOOLEAN DEFAULT FALSE,
        ORDER_VALUE_SUM DECIMAL(12,2) DEFAULT 0,
        DEVICE_TYPE VARCHAR(50),
        UTM_SOURCE VARCHAR(100),
        LANDING_PAGE_URL VARCHAR(500)
    )""",
    "EVENTS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.EVENTS (
        EVENT_ID VARCHAR(40) PRIMARY KEY,
        VISITOR_ID VARCHAR(40) NOT NULL,
        USER_ID VARCHAR(40),
        SESSION_ID VARCHAR(40) NOT NULL,
        EVENT_TS TIMESTAMP_NTZ NOT NULL,
        EVENT_TYPE VARCHAR(50) NOT NULL,
        EVENT_NAME VARCHAR(100),
        PRODUCT_ID VARCHAR(20),
        CATEGORY_ID VARCHAR(20),
        PAGE_URL VARCHAR(500),
        RECEIVED_TS TIMESTAMP_NTZ
    )""",
    "ORDERS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.ORDERS (
        ORDER_ID VARCHAR(40) PRIMARY KEY,
        USER_ID VARCHAR(40) NOT NULL,
        SESSION_ID VARCHAR(40),
        ORDER_TS TIMESTAMP_NTZ NOT NULL,
        STATUS VARCHAR(50) DEFAULT 'pending',
        SUBTOTAL_AMT DECIMAL(12,2),
        TAX_AMT DECIMAL(12,2),
        SHIPPING_AMT DECIMAL(12,2),
        DISCOUNT_AMT DECIMAL(12,2),
        TOTAL_AMT DECIMAL(12,2),
        ITEM_CNT INT,
        PAYMENT_METHOD VARCHAR(50),
        CREATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
        UPDATED_TS TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
    )""",
    "ORDER_ITEMS": f"""CREATE TABLE IF NOT EXISTS {prod_db}.{src}.ORDER_ITEMS (
        ORDER_ITEM_ID VARCHAR(40) PRIMARY KEY,
        ORDER_ID VARCHAR(40) NOT NULL,
        PRODUCT_ID VARCHAR(20) NOT NULL,
        SUPPLIER_ID VARCHAR(20),
        QUANTITY INT NOT NULL,
        UNIT_PRICE_AMT DECIMAL(10,2) NOT NULL,
        DISCOUNT_AMT DECIMAL(10,2) DEFAULT 0,
        TOTAL_AMT DECIMAL(12,2)
    )""",
}

for name, ddl in table_ddl.items():
    session.sql(ddl).collect()
    print(f"  {name}")

session.sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA {prod_db}.{src} TO ROLE {dev_role}").collect()
session.sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA {prod_db}.{src} TO ROLE {consumer_role}").collect()
print("\nAll source tables created")

## 8. Generate Synthetic Data

Populate tables with randomised but structurally consistent data. The `DATA_SCALE` parameter (default 0.01) controls volume. Set `FS_DATA_SCALE=1.0` for ~50k events.

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd

random.seed(42)
now = datetime.now()
base_ts = now - timedelta(days=365)

# --- Dimension tables ---
categories_data = [
    {"CATEGORY_ID": f"cat_{i:02d}", "CATEGORY_NAME": name, "IS_ACTIVE": True,
     "CREATED_TS": base_ts, "UPDATED_TS": base_ts}
    for i, name in enumerate([
        "ACCESSORIES", "BAGS", "BOOKS", "DRINKWARE", "GIFT",
        "HATS", "KIDS", "OFFICE", "OUTERWEAR", "PETS",
        "SME", "SNOW", "TEES",
    ], 1)
]
session.create_dataframe(pd.DataFrame(categories_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.CATEGORIES")

n_suppliers = 10
suppliers_data = [{
    "SUPPLIER_ID": f"sup_{i:03d}", "SUPPLIER_NAME": f"Supplier {i} Corp",
    "CONTACT_EMAIL": f"contact@supplier{i}.com",
    "COUNTRY": random.choice(["USA","USA","USA","CAN","GBR"]),
    "IS_ACTIVE": True, "RATING": round(random.uniform(3.0, 5.0), 2),
    "CREATED_TS": base_ts, "UPDATED_TS": base_ts,
} for i in range(1, n_suppliers+1)]
session.create_dataframe(pd.DataFrame(suppliers_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.SUPPLIERS")

n_products = 50
products_data = []
for i in range(1, n_products+1):
    cat = random.choice(categories_data)
    bp = round(random.uniform(10, 200), 2)
    products_data.append({
        "PRODUCT_ID": f"prod_{i:04d}",
        "PRODUCT_NAME": f"{cat['CATEGORY_NAME'].title()} Item {i}",
        "CATEGORY_ID": cat["CATEGORY_ID"], "BRAND": f"Brand{random.randint(1,10)}",
        "DESCRIPTION": f"A quality {cat['CATEGORY_NAME'].lower()} product.",
        "BASE_PRICE": bp, "CURRENT_PRICE": round(bp*random.uniform(0.8,1.0),2),
        "COST": round(bp*random.uniform(0.4,0.6),2),
        "SKU": f"{cat['CATEGORY_NAME'][:3]}-{i:04d}",
        "IS_ACTIVE": random.random() < 0.9,
        "CREATED_TS": base_ts, "UPDATED_TS": base_ts,
    })
session.create_dataframe(pd.DataFrame(products_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.PRODUCTS")

ps_data = []
for p in products_data:
    for j, s in enumerate(random.sample(suppliers_data, random.randint(1,3))):
        ps_data.append({
            "PRODUCT_ID": p["PRODUCT_ID"], "SUPPLIER_ID": s["SUPPLIER_ID"],
            "IS_PRIMARY": j==0, "LEAD_TIME_DAYS": random.randint(3,30),
            "UNIT_COST": round(p["COST"]*random.uniform(0.9,1.1),2),
            "MIN_ORDER_QTY": random.choice([1,5,10]), "CREATED_TS": base_ts,
        })
session.create_dataframe(pd.DataFrame(ps_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.PRODUCT_SUPPLIER")

print(f"Dimensions: {len(categories_data)} categories, {n_suppliers} suppliers, "
      f"{n_products} products, {len(ps_data)} product-supplier links")

In [ ]:
# --- Fact tables (scale-dependent) ---
n_households = max(30, int(1000 * DATA_SCALE))
hh_data = [{"HOUSEHOLD_ID": f"hh_{i:06d}",
    "CITY": random.choice(["San Francisco","New York","Seattle","Austin","Denver"]),
    "STATE": random.choice(["CA","NY","WA","TX","CO"]), "COUNTRY": "USA",
    "HOUSEHOLD_SIZE": random.randint(1,5),
    "INCOME_BRACKET": random.choice(["low","medium","medium","high","premium"]),
    "CREATED_TS": base_ts, "UPDATED_TS": base_ts,
} for i in range(1, n_households+1)]
session.create_dataframe(pd.DataFrame(hh_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.HOUSEHOLDS")

n_visitors = max(100, int(5000 * DATA_SCALE))
visitors_data = []
for i in range(1, n_visitors+1):
    fs_ts = base_ts + timedelta(days=random.randint(0,300))
    visitors_data.append({"VISITOR_ID": f"vis_{i:08d}",
        "FIRST_SEEN_TS": fs_ts,
        "LAST_SEEN_TS": fs_ts + timedelta(days=random.randint(1,60)),
        "FIRST_DEVICE_TYPE": random.choice(["mobile","desktop","tablet"]),
        "FIRST_UTM_SOURCE": random.choice(["google","direct","facebook","email",None]),
        "SESSION_CNT": random.randint(1,20), "PAGE_VIEW_CNT": random.randint(5,100),
        "CREATED_TS": fs_ts,
    })
session.create_dataframe(pd.DataFrame(visitors_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.VISITORS")

n_users = max(50, int(1000 * DATA_SCALE))
users_data = []
for i in range(1, n_users+1):
    vis = visitors_data[i-1] if i <= len(visitors_data) else random.choice(visitors_data)
    hh = random.choice(hh_data)
    reg = vis["FIRST_SEEN_TS"] + timedelta(days=random.randint(0,30))
    users_data.append({"USER_ID": f"usr_{i:08d}",
        "VISITOR_ID": vis["VISITOR_ID"], "HOUSEHOLD_ID": hh["HOUSEHOLD_ID"],
        "EMAIL": f"user{i}@example.com", "FIRST_NAME": f"First{i}", "LAST_NAME": f"Last{i}",
        "REGISTRATION_TS": reg,
        "LAST_LOGIN_TS": reg + timedelta(days=random.randint(0,200)),
        "IS_ACTIVE": random.random() < 0.85, "EMAIL_VERIFIED": random.random() < 0.8,
        "LOYALTY_TIER": random.choice(["Bronze","Silver","Gold","Platinum"]),
        "LOYALTY_POINTS": random.randint(0,5000),
        "PREFERRED_LANGUAGE": random.choice(["en","en","es","fr"]),
        "CREATED_TS": reg, "UPDATED_TS": reg + timedelta(days=random.randint(0,100)),
    })
session.create_dataframe(pd.DataFrame(users_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.USERS")

print(f"People: {n_households} households, {n_visitors} visitors, {n_users} users")

In [ ]:
# --- Sessions, Events, Orders ---
n_sessions = max(200, int(10000 * DATA_SCALE))
sessions_data = []
for i in range(1, n_sessions+1):
    user = random.choice(users_data) if random.random() < 0.6 else None
    vis = random.choice(visitors_data)
    start = base_ts + timedelta(seconds=random.randint(0, int(365*86400)))
    dur = random.randint(30, 1800)
    ec = random.randint(3, 20)
    ca = random.randint(0, 3)
    conv = random.random() < 0.03
    sessions_data.append({"SESSION_ID": f"sess_{i:08d}",
        "VISITOR_ID": vis["VISITOR_ID"],
        "USER_ID": user["USER_ID"] if user else None,
        "SESSION_START_TS": start,
        "SESSION_END_TS": start + timedelta(seconds=dur),
        "DURATION_SEC": dur, "EVENT_CNT": ec,
        "PAGE_VIEW_CNT": random.randint(1, ec),
        "PRODUCT_VIEW_DCNT": random.randint(0,5),
        "CART_ADD_CNT": ca,
        "CART_VALUE_SUM": round(ca*random.uniform(20,80),2) if ca else 0,
        "IS_CONVERTED": conv,
        "ORDER_VALUE_SUM": round(random.uniform(30,200),2) if conv else 0,
        "DEVICE_TYPE": random.choice(["mobile","desktop","tablet"]),
        "UTM_SOURCE": random.choice(["google","direct","facebook","email",None]),
        "LANDING_PAGE_URL": f"/products/prod_{random.randint(1,n_products):04d}",
    })
session.create_dataframe(pd.DataFrame(sessions_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.SESSIONS")

n_events = max(500, int(50000 * DATA_SCALE))
events_data = []
evt_types = [("Browsing","Products Searched"),("Product","Product Viewed"),
    ("Product","Product Clicked"),("Cart","Product Added"),
    ("Cart","Cart Viewed"),("Checkout","Checkout Started"),("Order","Order Completed")]
for i in range(1, n_events+1):
    sess = random.choice(sessions_data)
    et, en = random.choices(evt_types, weights=[15,30,20,10,8,5,2])[0]
    evt_ts = sess["SESSION_START_TS"] + timedelta(seconds=random.randint(0, max(1,sess["DURATION_SEC"])))
    events_data.append({"EVENT_ID": f"evt_{i:010d}",
        "VISITOR_ID": sess["VISITOR_ID"], "USER_ID": sess["USER_ID"],
        "SESSION_ID": sess["SESSION_ID"], "EVENT_TS": evt_ts,
        "EVENT_TYPE": et, "EVENT_NAME": en,
        "PRODUCT_ID": f"prod_{random.randint(1,n_products):04d}" if "Product" in en or "Cart" in en else None,
        "CATEGORY_ID": f"cat_{random.randint(1,13):02d}" if "Product" in en else None,
        "PAGE_URL": f"/page/{random.randint(1,100)}",
        "RECEIVED_TS": evt_ts + timedelta(milliseconds=random.randint(100,500)),
    })
session.create_dataframe(pd.DataFrame(events_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.EVENTS")

n_orders = max(30, int(3000 * DATA_SCALE))
orders_data = []
for i in range(1, n_orders+1):
    user = random.choice(users_data)
    ots = base_ts + timedelta(days=random.randint(0,350))
    sub = round(random.uniform(20,300),2)
    tax = round(sub*0.08,2)
    ship = round(random.uniform(5,15),2) if sub < 100 else 0
    disc = round(sub*0.1,2) if random.random() < 0.2 else 0
    orders_data.append({"ORDER_ID": f"ord_{i:08d}", "USER_ID": user["USER_ID"],
        "ORDER_TS": ots, "STATUS": random.choice(["confirmed","shipped","delivered"]),
        "SUBTOTAL_AMT": sub, "TAX_AMT": tax, "SHIPPING_AMT": ship,
        "DISCOUNT_AMT": disc, "TOTAL_AMT": round(sub+tax+ship-disc,2),
        "ITEM_CNT": random.randint(1,5),
        "PAYMENT_METHOD": random.choice(["credit_card","paypal","apple_pay"]),
        "CREATED_TS": ots, "UPDATED_TS": ots,
    })
session.create_dataframe(pd.DataFrame(orders_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.ORDERS")

oi_data = []
oi_id = 0
for order in orders_data:
    for _ in range(order["ITEM_CNT"]):
        oi_id += 1
        ps = random.choice(ps_data)
        qty = random.randint(1,3)
        price = round(random.uniform(10,150),2)
        d = round(price*random.uniform(0,0.15),2) if random.random() < 0.2 else 0
        oi_data.append({"ORDER_ITEM_ID": f"oi_{oi_id:010d}",
            "ORDER_ID": order["ORDER_ID"], "PRODUCT_ID": ps["PRODUCT_ID"],
            "SUPPLIER_ID": ps["SUPPLIER_ID"], "QUANTITY": qty,
            "UNIT_PRICE_AMT": price, "DISCOUNT_AMT": d,
            "TOTAL_AMT": round(qty*(price-d),2),
        })
session.create_dataframe(pd.DataFrame(oi_data)).write.mode("overwrite").save_as_table(
    f"{prod_db}.{src}.ORDER_ITEMS")

print(f"Activity: {n_sessions} sessions, {n_events} events, {n_orders} orders, {len(oi_data)} items")

## 9. Create Development Environment

Copy the production data to the development database. In a real deployment you might use clones or data sharing for zero-copy branching.

In [ ]:
# Create dev schemas
session.sql(f"USE ROLE {dev_role}").collect()
session.sql(f"USE DATABASE {dev_db}").collect()

for key, comment in schemas.items():
    sname = dev_cfg[key]
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {dev_db}.{sname} COMMENT = '{comment}'").collect()
    print(f"  {dev_db}.{sname}")

for s in schemas:
    sname = dev_cfg[s]
    for role in [consumer_role]:
        session.sql(f"GRANT USAGE ON SCHEMA {dev_db}.{sname} TO ROLE {role}").collect()
        session.sql(f"GRANT SELECT ON FUTURE TABLES IN SCHEMA {dev_db}.{sname} TO ROLE {role}").collect()
        session.sql(f"GRANT SELECT ON FUTURE VIEWS IN SCHEMA {dev_db}.{sname} TO ROLE {role}").collect()
        session.sql(f"GRANT SELECT ON FUTURE DYNAMIC TABLES IN SCHEMA {dev_db}.{sname} TO ROLE {role}").collect()

In [ ]:
# Copy all tables to dev
session.sql(f"USE ROLE {admin_role}").collect()
session.sql(f"USE WAREHOUSE {warehouse}").collect()

all_tables = ["CATEGORIES", "SUPPLIERS", "PRODUCTS", "PRODUCT_SUPPLIER",
              "HOUSEHOLDS", "VISITORS", "USERS", "SESSIONS", "EVENTS",
              "ORDERS", "ORDER_ITEMS"]

for t in all_tables:
    session.sql(f"CREATE OR REPLACE TABLE {dev_db}.{src}.{t} AS SELECT * FROM {prod_db}.{src}.{t}").collect()
    print(f"  Copied {t}")

session.sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA {dev_db}.{src} TO ROLE {dev_role}").collect()
session.sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA {dev_db}.{src} TO ROLE {consumer_role}").collect()
print("\nDev branch ready")

## 10. Enable ROW_TIMESTAMP & Deploy Generator Infrastructure

Enable `ROW_TIMESTAMP` on source tables so that `METADATA$ROW_LAST_COMMIT_TIME` records when each row was committed. This enables precise pipeline latency measurement in Notebook 05.

Also create the admin tables used by the incremental data generator: `GENERATION_CONFIG` (live-adjustable batch sizes), `GENERATION_STATE` (ID counters), and `GENERATION_LOG` (batch audit trail).

In [ ]:
# Enable ROW_TIMESTAMP on the 4 main source tables
session.sql(f"USE ROLE ACCOUNTADMIN").collect()
for table_name in ["SESSIONS", "EVENTS", "ORDERS", "ORDER_ITEMS"]:
    for db_name in [prod_db, dev_db]:
        session.sql(f"ALTER TABLE {db_name}.{src}.{table_name} SET ROW_TIMESTAMP = TRUE").collect()
    print(f"  {table_name}: ROW_TIMESTAMP enabled (both databases)")

# Create generator admin tables in the dev environment
session.sql(f"USE ROLE {admin_role}").collect()
session.sql(f"USE WAREHOUSE {warehouse}").collect()

from feature_definitions.generator import create_admin_tables, seed_state_from_existing_data

create_admin_tables(session, "DEV")
print("\nGenerator admin tables created: GENERATION_CONFIG, GENERATION_STATE, GENERATION_LOG")

counters = seed_state_from_existing_data(session, "DEV")
print(f"State seeded from existing data: {counters}")

## 11. Verification

Confirm all tables exist in both databases with expected row counts.

In [ ]:
session.sql(f"USE ROLE {admin_role}").collect()

for db_name in [prod_db, dev_db]:
    result = session.sql(f"""
        SELECT TABLE_NAME, ROW_COUNT
        FROM {db_name}.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{src}' AND TABLE_NAME NOT LIKE 'SNOWPARK%'
        ORDER BY TABLE_NAME
    """).collect()
    print(f"\n{db_name}.{src}:")
    for r in result:
        print(f"  {r['TABLE_NAME']:25s} {r['ROW_COUNT']:>8,} rows")

print("\n✅ Platform setup complete! Proceed to Notebook 01: Feature Engineering.")